# 044: Python Strings — Complete A–Z Method Reference, Unicode Internals, Interning, and Formatting Engines

## 1. WHAT IS A PYTHON STRING?

A Python `str` is an **immutable sequence of Unicode code points**. Every character in a Python 3 string is a Unicode character, not a raw byte.

### CPython Internal Representation (`unicodeobject.h`) — PEP 393 Flexible String Representation

CPython 3.3+ uses a **compact** representation that adapts storage per character:
- **Latin-1 (1 byte/char)**: If all code points are ≤ U+00FF (ASCII + Latin supplements).
- **UCS-2 (2 bytes/char)**: If any code point is > U+00FF but ≤ U+FFFF (Basic Multilingual Plane).
- **UCS-4 (4 bytes/char)**: If any code point is > U+FFFF (emojis, rare CJK, historic scripts).

This means a string of pure ASCII characters uses only **1 byte per character** in memory (very efficient), but inserting a single emoji forces the entire string to UCS-4 (4 bytes per char).

### String Immutability
- Once created, a string object cannot be modified. Every "modification" operation returns a **new** string.
- This is why repeated string concatenation in a loop (`s += "x"`) is $O(n^2)$ — each iteration creates a new string and copies all existing characters. Use `"".join(parts)` instead.

### String Interning
- CPython automatically **interns** (caches and reuses) certain strings:
  - String literals that look like valid identifiers (e.g., `"hello"`, `"my_var"`).
  - Strings of length 0 or 1.
  - Compile-time constant expressions.
- Interned strings are compared with `is` (pointer comparison, $O(1)$) instead of `==` (character-by-character, $O(n)$).
- You can force interning with `sys.intern(s)`.

---


In [ ]:
import sys

print("=" * 70)
print("SECTION 1: String Memory Layout — PEP 393 Flexible Representation")
print("=" * 70)

# ASCII-only (1 byte/char)
ascii_str = "hello"
print(f"'{ascii_str}' size: {sys.getsizeof(ascii_str)} bytes ({sys.getsizeof(ascii_str) - 49} content bytes for {len(ascii_str)} chars)")

# Latin-1 (1 byte/char with accents)
latin_str = "café"
print(f"'{latin_str}' size: {sys.getsizeof(latin_str)} bytes")

# BMP character (2 bytes/char)
cjk_str = "你好"
print(f"'{cjk_str}' size: {sys.getsizeof(cjk_str)} bytes")

# Emoji (4 bytes/char) — forces UCS-4
emoji_str = "hi😀"
print(f"'{emoji_str}' size: {sys.getsizeof(emoji_str)} bytes")
# Compare: "hi" alone is much smaller
print(f"'hi' size:  {sys.getsizeof('hi')} bytes")
print("[Key Insight] A single emoji forces all chars to 4 bytes each!")



In [ ]:
# Interning demonstration
print("\n" + "=" * 70)
print("SECTION 1b: String Interning")
print("=" * 70)

a = "hello"
b = "hello"
print(f"'hello' is 'hello':          {a is b}")  # True — interned

# Dynamic strings are NOT interned
c = "hel" + "lo"  # Compile-time constant folding → interned
d = "".join(["h", "e", "l", "l", "o"])  # Runtime construction → NOT interned
print(f"compile-time concat is:      {a is c}")
print(f"runtime join is:             {a is d}")

# Force interning
e = sys.intern(d)
print(f"sys.intern() is original:    {a is e}")  # True — now interned



---
## 2. COMPLETE A–Z STRING METHOD REFERENCE

Python strings have **47 built-in methods**. Every single one is documented below.

---
### 2.1 CASE CONVERSION METHODS

| Method | Description | Time |
|:---|:---|:---:|
| `str.upper()` | All chars to uppercase | $O(n)$ |
| `str.lower()` | All chars to lowercase | $O(n)$ |
| `str.capitalize()` | First char uppercase, rest lowercase | $O(n)$ |
| `str.title()` | First char of each word uppercase | $O(n)$ |
| `str.swapcase()` | Swap upper↔lower | $O(n)$ |
| `str.casefold()` | Aggressive lowercase (for case-insensitive comparison) | $O(n)$ |


In [ ]:
print("\n" + "=" * 70)
print("CASE CONVERSION METHODS")
print("=" * 70)

s = "hello WORLD Python"
print(f"upper():                     '{s.upper()}'")
print(f"lower():                     '{s.lower()}'")
print(f"capitalize():                '{s.capitalize()}'")
print(f"title():                     '{s.title()}'")
print(f"swapcase():                  '{s.swapcase()}'")

# casefold vs lower — casefold handles special Unicode cases
german = "straße"  # ß → ss in casefold
print(f"'straße'.lower():            '{german.lower()}'")
print(f"'straße'.casefold():         '{german.casefold()}'")  # 'strasse'



### 2.2 SEARCH & FIND METHODS

| Method | Description | Raises |
|:---|:---|:---:|
| `str.find(sub, start, end)` | Index of first occurrence, or -1 | No |
| `str.rfind(sub, start, end)` | Index of last occurrence, or -1 | No |
| `str.index(sub, start, end)` | Like find(), but raises ValueError | Yes |
| `str.rindex(sub, start, end)` | Like rfind(), but raises ValueError | Yes |
| `str.count(sub, start, end)` | Number of non-overlapping occurrences | No |


In [ ]:
print("\n" + "=" * 70)
print("SEARCH & FIND METHODS")
print("=" * 70)

text = "banana banana banana"
print(f"find('banana'):              {text.find('banana')}")
print(f"find('banana', 1):           {text.find('banana', 1)}")
print(f"rfind('banana'):             {text.rfind('banana')}")
print(f"find('mango'):               {text.find('mango')}")  # -1

print(f"count('banana'):             {text.count('banana')}")
print(f"count('an'):                 {text.count('an')}")

try:
    text.index("mango")
except ValueError as e:
    print(f"index('mango'):              ValueError: {e}")



### 2.3 BOOLEAN TEST METHODS (is* family)

| Method | Tests for | Example True |
|:---|:---|:---|
| `str.isalpha()` | All alphabetic | `"Hello"` |
| `str.isdigit()` | All digits | `"123"` |
| `str.isalnum()` | All alphanumeric | `"abc123"` |
| `str.isspace()` | All whitespace | `"  \t\n"` |
| `str.isupper()` | All cased chars uppercase | `"HELLO"` |
| `str.islower()` | All cased chars lowercase | `"hello"` |
| `str.istitle()` | Title case | `"Hello World"` |
| `str.isnumeric()` | All numeric (includes fractions, etc.) | `"½"` |
| `str.isdecimal()` | All decimal digits | `"0123"` |
| `str.isascii()` | All ASCII (U+0000 to U+007F) | `"hello"` |
| `str.isidentifier()` | Valid Python identifier | `"my_var"` |
| `str.isprintable()` | All printable | `"hello"` |


In [ ]:
print("\n" + "=" * 70)
print("BOOLEAN TEST METHODS (is* family)")
print("=" * 70)

tests = [
    ("'Hello'", "Hello"),
    ("'123'", "123"),
    ("'abc123'", "abc123"),
    ("'  \\t'", "  \t"),
    ("'HELLO'", "HELLO"),
    ("'hello'", "hello"),
    ("''", ""),
    ("'½'", "½"),
]

for label, s in tests:
    if s:  # Skip empty string for brevity
        print(f"{label:>12}.isalpha()={s.isalpha()}, .isdigit()={s.isdigit()}, "
              f".isalnum()={s.isalnum()}, .isspace()={s.isspace()}")

# Edge case: empty string returns False for ALL is* methods
print(f"\n''.isalpha():                {str.isalpha('')}")
print(f"''.isdigit():                {str.isdigit('')}")

# isidentifier
print(f"'my_var'.isidentifier():     {'my_var'.isidentifier()}")
print(f"'2fast'.isidentifier():      {'2fast'.isidentifier()}")  # Starts with digit
print(f"'class'.isidentifier():      {'class'.isidentifier()}")  # True! (doesn't check keywords)

# isascii (Python 3.7+)
print(f"'hello'.isascii():           {'hello'.isascii()}")
print(f"'café'.isascii():            {'café'.isascii()}")  # False



### 2.4 STRING MODIFICATION METHODS

| Method | Description |
|:---|:---|
| `str.strip([chars])` | Remove leading and trailing chars |
| `str.lstrip([chars])` | Remove leading chars only |
| `str.rstrip([chars])` | Remove trailing chars only |
| `str.replace(old, new, count)` | Replace occurrences |
| `str.expandtabs(tabsize)` | Replace tabs with spaces |
| `str.zfill(width)` | Pad with zeros on the left |
| `str.center(width, fillchar)` | Center-align with padding |
| `str.ljust(width, fillchar)` | Left-align with padding |
| `str.rjust(width, fillchar)` | Right-align with padding |


In [ ]:
print("\n" + "=" * 70)
print("STRING MODIFICATION METHODS")
print("=" * 70)

s = "  hello world  "
print(f"strip():                     '{s.strip()}'")
print(f"lstrip():                    '{s.lstrip()}'")
print(f"rstrip():                    '{s.rstrip()}'")

# strip with specific characters
s2 = "***hello***"
print(f"strip('*'):                  '{s2.strip('*')}'")

# replace
print(f"replace:                     '{'hello world'.replace('world', 'Python')}'")
print(f"replace(count=1):            '{'aaa'.replace('a', 'b', 1)}'")

# Padding
print(f"zfill:                       '{'42'.zfill(6)}'")
print(f"center:                      '{'hi'.center(10, '-')}'")
print(f"ljust:                       '{'hi'.ljust(10, '.')}'")
print(f"rjust:                       '{'hi'.rjust(10, '.')}'")



### 2.5 SPLIT AND JOIN METHODS

| Method | Description |
|:---|:---|
| `str.split(sep, maxsplit)` | Split by separator (default: whitespace) |
| `str.rsplit(sep, maxsplit)` | Split from the right |
| `str.splitlines(keepends)` | Split by line boundaries |
| `str.partition(sep)` | Split into 3-tuple at first occurrence |
| `str.rpartition(sep)` | Split into 3-tuple at last occurrence |
| `str.join(iterable)` | Join iterable elements with separator |


In [ ]:
print("\n" + "=" * 70)
print("SPLIT AND JOIN METHODS")
print("=" * 70)

s = "one two three four five"
print(f"split():                     {s.split()}")
print(f"split(' ', 2):               {s.split(' ', 2)}")
print(f"rsplit(' ', 2):              {s.rsplit(' ', 2)}")

# splitlines
text = "line1\nline2\r\nline3\rline4"
print(f"splitlines():                {text.splitlines()}")
print(f"splitlines(True):            {text.splitlines(True)}")

# partition
s = "hello=world=python"
print(f"partition('='):              {s.partition('=')}")
print(f"rpartition('='):             {s.rpartition('=')}")

# join
print(f"join:                        {', '.join(['a', 'b', 'c'])}")
print(f"join with path:              {'/'.join(['usr', 'local', 'bin'])}")

# Performance: join vs +=
import time

N = 50000
t0 = time.perf_counter()
parts = []
for i in range(N):
    parts.append(str(i))
result_join = ",".join(parts)
t1 = time.perf_counter()
join_time = (t1 - t0) * 1000

t0 = time.perf_counter()
result_concat = ""
for i in range(N):
    result_concat += str(i) + ","
t1 = time.perf_counter()
concat_time = (t1 - t0) * 1000

print(f"\njoin (50K items):             {join_time:.1f} ms")
print(f"+= concat (50K items):        {concat_time:.1f} ms")
print(f"join is {concat_time / (join_time + 0.001):.1f}x faster")



### 2.6 STARTSWITH AND ENDSWITH

| Method | Description |
|:---|:---|
| `str.startswith(prefix, start, end)` | Tests if string starts with prefix. Accepts tuple of prefixes. |
| `str.endswith(suffix, start, end)` | Tests if string ends with suffix. Accepts tuple of suffixes. |


In [ ]:
print("\n" + "=" * 70)
print("STARTSWITH AND ENDSWITH")
print("=" * 70)

filename = "document.pdf"
print(f"endswith('.pdf'):             {filename.endswith('.pdf')}")

# Tuple of options (check multiple extensions)
print(f"endswith(('.pdf','.doc')):    {filename.endswith(('.pdf', '.doc'))}")
print(f"endswith(('.txt','.csv')):    {filename.endswith(('.txt', '.csv'))}")

url = "https://example.com"
print(f"startswith('https'):         {url.startswith('https')}")
print(f"startswith(('http','ftp')):  {url.startswith(('http', 'ftp'))}")



### 2.7 ENCODING AND DECODING

| Method | Description |
|:---|:---|
| `str.encode(encoding, errors)` | Convert str → bytes |
| `bytes.decode(encoding, errors)` | Convert bytes → str |


In [ ]:
print("\n" + "=" * 70)
print("ENCODING AND DECODING")
print("=" * 70)

s = "Hello, 世界! 🌍"
# UTF-8 encoding (variable length: 1-4 bytes per char)
utf8 = s.encode("utf-8")
print(f"UTF-8 bytes:                 {utf8}")
print(f"UTF-8 length:                {len(utf8)} bytes for {len(s)} chars")

# UTF-16 encoding
utf16 = s.encode("utf-16")
print(f"UTF-16 length:               {len(utf16)} bytes")

# ASCII encoding — fails on non-ASCII
try:
    s.encode("ascii")
except UnicodeEncodeError as e:
    print(f"ASCII encode error:          {e.reason} at position {e.start}")

# Error handling strategies
print(f"errors='ignore':             {s.encode('ascii', errors='ignore')}")
print(f"errors='replace':            {s.encode('ascii', errors='replace')}")

# Decode back
decoded = utf8.decode("utf-8")
print(f"Decoded back:                '{decoded}'")
print(f"Round-trip match:            {decoded == s}")



### 2.8 FORMAT METHODS

Python has three string formatting systems:
1. **%-formatting** (old style): `"Hello %s" % name`
2. **str.format()**: `"Hello {}".format(name)`
3. **f-strings** (Python 3.6+): `f"Hello {name}"` — compiled to bytecode, fastest.


In [ ]:
print("\n" + "=" * 70)
print("STRING FORMATTING ENGINES")
print("=" * 70)

name, age, pi = "Alice", 30, 3.14159

# %-formatting
print(f"%-format:                    {'Hello %s, age %d' % (name, age)}")

# str.format()
print(f"str.format():                {'Hello {}, age {}'.format(name, age)}")
print(f"Named format:                {'Hello {name}, pi={pi:.2f}'.format(name=name, pi=pi)}")

# f-strings (fastest)
print(f"f-string:                    Hello {name}, pi={pi:.4f}")

# Format spec mini-language
num = 42
print(f"Binary:                      {num:08b}")
print(f"Hex:                         {num:#06x}")
print(f"Thousands separator:         {1234567:,}")
print(f"Percentage:                  {0.856:.1%}")
print(f"Left-align:                  '{name:<10}'")
print(f"Right-align:                 '{name:>10}'")
print(f"Center-align:                '{name:^10}'")

# Performance comparison
import time

N = 200_000
t0 = time.perf_counter()
for _ in range(N):
    s = f"Hello {name}, age {age}"
t1 = time.perf_counter()
fstring_time = (t1 - t0) * 1000

t0 = time.perf_counter()
for _ in range(N):
    s = "Hello {}, age {}".format(name, age)
t1 = time.perf_counter()
format_time = (t1 - t0) * 1000

t0 = time.perf_counter()
for _ in range(N):
    s = "Hello %s, age %d" % (name, age)
t1 = time.perf_counter()
percent_time = (t1 - t0) * 1000

print(f"\nf-string (200K):             {fstring_time:.1f} ms")
print(f"str.format (200K):           {format_time:.1f} ms")
print(f"%-format (200K):             {percent_time:.1f} ms")



### 2.9 REMAINING METHODS

| Method | Description |
|:---|:---|
| `str.maketrans(x, y, z)` | Create translation table |
| `str.translate(table)` | Apply translation table |
| `str.removeprefix(prefix)` | (3.9+) Remove prefix if present |
| `str.removesuffix(suffix)` | (3.9+) Remove suffix if present |


In [ ]:
print("\n" + "=" * 70)
print("TRANSLATION AND PREFIX/SUFFIX REMOVAL")
print("=" * 70)

# maketrans + translate (character-level substitution)
table = str.maketrans("aeiou", "12345")
print(f"translate vowels→digits:     {'hello world'.translate(table)}")

# Remove specific characters
table2 = str.maketrans("", "", "aeiou")
print(f"Remove vowels:               {'hello world'.translate(table2)}")

# removeprefix / removesuffix (Python 3.9+)
try:
    print(f"removeprefix('Hello'):       {'HelloWorld'.removeprefix('Hello')}")
    print(f"removesuffix('.py'):          {'script.py'.removesuffix('.py')}")
    print(f"removeprefix (no match):     {'HelloWorld'.removeprefix('Bye')}")
except AttributeError:
    print("removeprefix/removesuffix requires Python 3.9+")



---
## 3. SLICING OPTIMIZATION

- String slicing creates a **new** string object. $O(k)$ where $k$ = slice length.
- For single-character access, CPython caches all Latin-1 characters (U+0000 to U+00FF), so `s[i]` returns a cached singleton.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: Slicing & Single-Char Caching")
print("=" * 70)

s = "Hello, World!"
print(f"s[0:5]:                      '{s[0:5]}'")
print(f"s[::-1]:                     '{s[::-1]}'")
print(f"s[7:]:                       '{s[7:]}'")

# Single character caching
a = "h"
b = "h"
print(f"'h' is 'h':                  {a is b}")  # True — cached Latin-1 singleton

# But sliced chars may or may not be interned
c = "hello"[0]
print(f"'hello'[0] is 'h':           {c is a}")



---
## 4. FAANG & RESEARCH INTERVIEW QUESTIONS

### Q1: Why is string concatenation with `+=` in a loop $O(n^2)$?
**A**: Strings are immutable. Each `s += "x"` creates a **new** string of length `len(s) + 1`, copying all existing characters. After $n$ iterations, total work is $1 + 2 + 3 + \ldots + n = O(n^2)$. Use `"".join(parts)` which allocates the final string once and copies each part exactly once: $O(n)$.

### Q2: What is string interning and when does CPython do it?
**A**: Interning stores a single copy of each distinct string and reuses it. CPython automatically interns: (1) string literals that look like identifiers, (2) strings of length 0 or 1, (3) compile-time constant expressions. You can force interning with `sys.intern()`. Interned strings enable $O(1)$ comparison via pointer identity (`is`).

### Q3: Explain PEP 393 Flexible String Representation.
**A**: Before PEP 393, Python used either UCS-2 or UCS-4 for ALL characters in a string. PEP 393 (CPython 3.3+) uses the narrowest representation that fits all characters: 1 byte/char for Latin-1, 2 bytes/char for BMP, 4 bytes/char for full Unicode. A single emoji in an otherwise ASCII string forces the entire string to 4 bytes/char.

### Q4: Why is `str.casefold()` preferred over `str.lower()` for case-insensitive comparison?
**A**: `casefold()` handles Unicode special cases. For example, the German `ß` (eszett) is case-folded to `ss`, while `lower()` leaves it unchanged. For case-insensitive comparison of international text, `casefold()` is the correct choice.

### Q5: What is the difference between `str.split()` and `str.split(' ')`?
**A**: `split()` with no arguments splits on any whitespace (spaces, tabs, newlines) and removes empty strings from the result. `split(' ')` splits only on space characters and preserves empty strings. Example: `"  a  b  ".split()` → `['a', 'b']`, but `"  a  b  ".split(' ')` → `['', '', 'a', '', 'b', '', '']`.


In [ ]:
# Demonstrating Q5
print("\n" + "=" * 70)
print("INTERVIEW DEMO: split() vs split(' ')")
print("=" * 70)

s = "  hello   world  "
print(f"split():                     {s.split()}")
print(f"split(' '):                  {s.split(' ')}")
